# Import Libraries

In [55]:
import boto3
import pandas as pd
import numpy as np
import sklearn
import sagemaker
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler

# Checking Up S3 Buckets and Locate Datasets

In [56]:
s3_resource = boto3.resource('s3')
bucket = s3_resource.Bucket('ads508-team1-sephora')

for items in bucket.objects.all():
    print(items.key)

athena/
athena/staging/
athena/staging/2db1a0d5-fa3b-49d6-92c3-f34da040eb33.txt
athena/staging/63402cb0-ca6f-40ed-bc68-e7980d01f348.txt
athena/staging/c427eba1-fadc-4d2a-ac8f-4b85a6e59415.txt
athena/staging/cfa0370a-881e-46f2-aa8e-0f8fa10ac191.txt
curated/
curated/products/
curated/products/products.parquet
curated/reviews/
curated/reviews/reviews.parquet
raw/
raw/products/
raw/products/product_info.csv
raw/reviews/
raw/reviews/reviews_0-250.csv
raw/reviews/reviews_1250-end.csv
raw/reviews/reviews_250-500.csv
raw/reviews/reviews_500-750.csv
raw/reviews/reviews_750-1250.csv


# Load the Product Info Dataset

In [57]:
# Define the S3 path
s3_path = 's3://ads508-team1-sephora/curated/products/products.parquet'

# Load the parquet directly into a DataFrame
df_products = pd.read_parquet(s3_path)

# Display the first few rows
df_products.head().T

,0,1,2,3,4
product_id,P473671,P473668,P473662,P473660,P473658
product_name,Fragrance Discovery Set,La Habana Eau de Parfum,Rainbow Bar Eau de Parfum,Kasbah Eau de Parfum,Purple Haze Eau de Parfum
brand_id,6342,6342,6342,6342,6342
brand_name,19-69,19-69,19-69,19-69,19-69
loves_count,6320,3827,3253,3018,2691
rating,3.6364,4.1538,4.25,4.4762,3.2308
reviews,11.0,13.0,16.0,21.0,13.0
size,None,3.4 oz/ 100 mL,3.4 oz/ 100 mL,3.4 oz/ 100 mL,3.4 oz/ 100 mL
variation_type,None,Size + Concentration + Formulation,Size + Concentration + Formulation,Size + Concentration + Formulation,Size + Concentration + Formulation
variation_value,None,3.4 oz/ 100 mL,3.4 oz/ 100 mL,3.4 oz/ 100 mL,3.4 oz/ 100 mL


# Checking Out the Product Dataset Shape

In [58]:
# Product File Information
df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8494 entries, 0 to 8493
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_id          8494 non-null   object 
 1   product_name        8494 non-null   object 
 2   brand_id            8494 non-null   int64  
 3   brand_name          8494 non-null   object 
 4   loves_count         8494 non-null   int64  
 5   rating              8216 non-null   float64
 6   reviews             8216 non-null   float64
 7   size                6863 non-null   object 
 8   variation_type      7050 non-null   object 
 9   variation_value     6896 non-null   object 
 10  variation_desc      1250 non-null   object 
 11  ingredients         7549 non-null   object 
 12  price_usd           8494 non-null   float64
 13  value_price_usd     451 non-null    float64
 14  sale_price_usd      270 non-null    float64
 15  limited_edition     8494 non-null   int64  
 16  new   

In [59]:
# Product file shape
df_products.shape

(8494, 27)

In [60]:
# Profuct file description
df_products.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
product_id,8494,8494,P505461,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_name,8494,8415,Hand Wash,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand_id,8494.0,NaN,NaN,NaN,5422.440546,1709.595957,1063.0,5333.0,6157.5,6328.0,8020.0
brand_name,8494,304,SEPHORA COLLECTION,352,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loves_count,8494.0,NaN,NaN,NaN,29179.565929,66092.12259,0.0,3758.0,9880.0,26841.25,1401068.0
rating,8216.0,NaN,NaN,NaN,4.194513,0.516694,1.0,3.981725,4.28935,4.530525,5.0
reviews,8216.0,NaN,NaN,NaN,448.545521,1101.982529,1.0,26.0,122.0,418.0,21281.0
size,6863,2055,1.7 oz/ 50 mL,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
variation_type,7050,7,Size,4043,NaN,NaN,NaN,NaN,NaN,NaN,NaN
variation_value,6896,2729,1.7 oz/ 50 mL,374,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
# Checking missing data points
df_products.isnull().sum()

product_id               0
product_name             0
brand_id                 0
brand_name               0
loves_count              0
rating                 278
reviews                278
size                  1631
variation_type        1444
variation_value       1598
variation_desc        7244
ingredients            945
price_usd                0
value_price_usd       8043
sale_price_usd        8224
limited_edition          0
new                      0
online_only              0
out_of_stock             0
sephora_exclusive        0
highlights            2207
primary_category         0
secondary_category       8
tertiary_category      990
child_count              0
child_max_price       5740
child_min_price       5740
dtype: int64

In [62]:
# Checking missing data points percentage
df_products.isnull().mean()*100

product_id             0.000000
product_name           0.000000
brand_id               0.000000
brand_name             0.000000
loves_count            0.000000
rating                 3.272899
reviews                3.272899
size                  19.201789
variation_type        17.000235
variation_value       18.813280
variation_desc        85.283730
ingredients           11.125500
price_usd              0.000000
value_price_usd       94.690370
sale_price_usd        96.821286
limited_edition        0.000000
new                    0.000000
online_only            0.000000
out_of_stock           0.000000
sephora_exclusive      0.000000
highlights            25.983047
primary_category       0.000000
secondary_category     0.094184
tertiary_category     11.655286
child_count            0.000000
child_max_price       67.577113
child_min_price       67.577113
dtype: float64

# Imputation of Missing Value

## 1) Fill in mean rating for missing rating

In [63]:
# Fill in mean Rating for missing Rating
# Calculate the mean of the rating column
mean_rating = df_products['rating'].mean()

# Fill missing values in 'rating' with the mean
df_products['rating'] = df_products['rating'].fillna(mean_rating)

# Verify that there are no more missing values in 'rating'
print(f"Missing ratings remaining: {df_products['rating'].isnull().sum()}")

Missing ratings remaining: 0


## 2) Fill in zero for missing reviews

In [64]:
# Fill in zero for missing reviews
# Fill missing values in 'reviews' with 0
df_products['reviews'] = df_products['reviews'].fillna(0)

# Verify that there are no more missing values in 'reviews'
print(f"Missing reviews remaining: {df_products['reviews'].isnull().sum()}")

Missing reviews remaining: 0


## 3) Fill in NA for missing size, variation_type, variation_value, variation_desc, ingredients, highlights, secondary_category, tertiary_category

In [65]:
# Fill in NA for missing size, variation_type, variation_value, variation_desc, ingredients, highlights, secondary_category, tertiary_category
# List of columns to fill with "NA"
cols_to_fix = [
    'size', 'variation_type', 'variation_value', 
    'variation_desc', 'ingredients', 'highlights', 'secondary_category', 'tertiary_category'
]

# Fill missing values in these columns with "NA"
df_products[cols_to_fix] = df_products[cols_to_fix].fillna("NA")

# Verify that no missing values remain in these columns
print(df_products[cols_to_fix].isnull().sum())

size                  0
variation_type        0
variation_value       0
variation_desc        0
ingredients           0
highlights            0
secondary_category    0
tertiary_category     0
dtype: int64


## 4) Fill in price_usd for missing value in value_price_usd and sale_price_usd

In [66]:
# Fill in price_usd for missing value in value_price_usd and sale_price_usd
# Fill missing value_price_usd with the standard price_usd
df_products['value_price_usd'] = df_products['value_price_usd'].fillna(df_products['price_usd'])

# Fill missing sale_price_usd with the standard price_usd
df_products['sale_price_usd'] = df_products['sale_price_usd'].fillna(df_products['price_usd'])

# Verify that no missing values remain in these price columns
print(f"Missing value_price_usd: {df_products['value_price_usd'].isnull().sum()}")
print(f"Missing sale_price_usd: {df_products['sale_price_usd'].isnull().sum()}")

Missing value_price_usd: 0
Missing sale_price_usd: 0


## 5) Fill in zero for missing child_max_price and child_min_price

In [67]:
# Fill in zero for missing child_max_price and child_min_price
# Fill missing values for child price range columns with 0
df_products['child_max_price'] = df_products['child_max_price'].fillna(0)
df_products['child_min_price'] = df_products['child_min_price'].fillna(0)

# Verify the changes
print(f"Missing child_max_price: {df_products['child_max_price'].isnull().sum()}")
print(f"Missing child_min_price: {df_products['child_min_price'].isnull().sum()}")

Missing child_max_price: 0
Missing child_min_price: 0


## 6) Confirmation for no missing values

In [68]:
# Checking missing data points percentage
df_products.isnull().mean()*100

product_id            0.0
product_name          0.0
brand_id              0.0
brand_name            0.0
loves_count           0.0
rating                0.0
reviews               0.0
size                  0.0
variation_type        0.0
variation_value       0.0
variation_desc        0.0
ingredients           0.0
price_usd             0.0
value_price_usd       0.0
sale_price_usd        0.0
limited_edition       0.0
new                   0.0
online_only           0.0
out_of_stock          0.0
sephora_exclusive     0.0
highlights            0.0
primary_category      0.0
secondary_category    0.0
tertiary_category     0.0
child_count           0.0
child_max_price       0.0
child_min_price       0.0
dtype: float64

# Feature Engineering

## 1) Initial price feature engineering

In [69]:
# 1. Log transformation of price to handle skewness
df_products['price_log'] = np.log1p(df_products['price_usd'])

# 2. Identify if a product is actually on sale 
# (Comparing sale_price_usd to the original price_usd)
df_products['is_on_sale'] = (df_products['sale_price_usd'] < df_products['price_usd']).astype(int)

# 3. Calculate the discount percentage
# If no sale, this will result in 0.0
df_products['discount_pct'] = (df_products['price_usd'] - df_products['sale_price_usd']) / df_products['price_usd']

# 4. Calculate the value ratio (Value for money)
# (value_price_usd / price_usd)
df_products['value_ratio'] = df_products['value_price_usd'] / df_products['price_usd']

# Display the new features for the first few rows
df_products[['price_usd', 'price_log','sale_price_usd', 'is_on_sale', 'discount_pct', 'value_ratio']].head()

,price_usd,price_log,sale_price_usd,is_on_sale,discount_pct,value_ratio
0,35.0,3.583519,35.0,0,0.0,1.0
1,195.0,5.278115,195.0,0,0.0,1.0
2,195.0,5.278115,195.0,0,0.0,1.0
3,195.0,5.278115,195.0,0,0.0,1.0
4,195.0,5.278115,195.0,0,0.0,1.0


Log transformation of price_usd (price_log): Applied np.log1p to normalize the skewed price distribution and reduce the impact of extreme outliers (like the $1,900 items).

Price sensitivity & promotion tracking (is_on_sale & discount_pct): Created binary indicators and percentage calculations to capture whether a product is discounted relative to its original price.

Perceived value mapping (value_ratio): Calculated the ratio of value_price_usd to price_usd to quantify "value for money," especially for gift sets and bundles.

## 2) Price Bucketing (Business-friendly)

In [70]:
# Define price segments based on the distribution of price_usd
# Min: $3.0 | 25%: $25.0 | 50%: $35.0 | 75%: $58.0 | Max: $1,900.0
bins = [0, 25, 50, 100, 200, np.inf]
labels = ['budget', 'low_mid', 'mid', 'premium', 'luxury']

# Create the price_bucket feature
df_products['price_bucket'] = pd.cut(df_products['price_usd'], bins=bins, labels=labels)

# Verify the distribution of the new buckets
print(df_products['price_bucket'].value_counts())

price_bucket
low_mid    3707
budget     2314
mid        1575
premium     723
luxury      175
Name: count, dtype: int64


Easier for Models & Business Interpretation: Converting exact prices into ranges reduces noise for machine learning algorithms and aligns with how retail stakeholders view product "tiers."

Hypothesis Testing: This allows us to move beyond individual price points and answer high-level business questions, such as: "Do premium products ($100–$200) maintain higher average ratings or 'loves' counts than budget items?"

## 3) Brand Features

In [71]:
# Identify the top 20 brands by product count
top_brands = df_products['brand_name'].value_counts().nlargest(20).index

# Group any brand not in the top 20 as 'Other' to reduce category cardinality
df_products['brand_group'] = df_products['brand_name'].apply(lambda x: x if x in top_brands else 'Other')

# Check the distribution of the new brand groups
print(df_products['brand_group'].value_counts())

brand_group
Other                      6251
SEPHORA COLLECTION          352
CLINIQUE                    179
Dior                        136
tarte                       131
NEST New York               115
Bumble and bumble           110
Kérastase                   108
TOM FORD                    100
Charlotte Tilbury            99
Anastasia Beverly Hills      95
Oribe                        94
Moroccanoil                  91
Jo Malone London             88
Hourglass                    87
Fenty Beauty by Rihanna      82
Lancôme                      81
Kiehl's Since 1851           75
Benefit Cosmetics            74
MAKE UP FOR EVER             74
Drybar                       72
Name: count, dtype: int64


Brand Categorization (brand_group): We identified the top 20 brands by product count and consolidated all remaining brands into a single "Other" category.

Solves High-Cardinality: Prevents "noise" in machine learning models by reducing hundreds of unique brand names—many with only 1 or 2 products—into a manageable set of features.

Captures Brand Reputation: Isolates the "Power Players" (e.g., SEPHORA COLLECTION, Clinique, Dior) where we have enough data to draw statistically significant conclusions about brand performance.

Dimensionality Reduction: Streamlines future visualizations and one-hot encoding, ensuring that our analysis (if we need to) focuses on the most influential market competitors rather than long-tail outliers.

## 4) Popularity Features

In [72]:
# 1. Log transformation of loves_count
# Helps manage the massive range (0 to 1.4M+) and reduces skewness
df_products['log_loves'] = np.log1p(df_products['loves_count'])

# 2. Loves per Price (Value Sentiment)
# Measures how much 'love' a product gets per dollar spent
df_products['loves_per_price'] = df_products['loves_count'] / df_products['price_usd']

# 3. Popularity Score (Interaction weighted by Rating)
# A simple way to combine volume (reviews) and sentiment (rating)
df_products['popularity_index'] = df_products['reviews'] * df_products['rating']

# Display the results for high-interaction products
df_products[['product_name', 'loves_count', 'log_loves', 'loves_per_price', 'popularity_index']].sort_values(by='loves_count', ascending=False).head()

,product_name,loves_count,log_loves,loves_per_price,popularity_index
6242,Soft Pinch Liquid Blush,1401068,14.152746,60916.000000,21466.9948
5249,Radiant Creamy Concealer,1153594,13.958394,36049.812500,55517.1960
4431,Lip Sleeping Mask Intense Hydration with Vitam...,1081315,13.893689,45054.791667,70126.1944
6434,Cream Lip Stain Liquid Lipstick,1029051,13.844149,68603.400000,48000.6311
2523,Gloss Bomb Universal Lip Luminizer,968317,13.783316,46110.333333,56258.8552


Popularity & Engagement Metrics (log_loves, loves_per_price, popularity_index): We engineered several features to quantify product "hype" and consumer sentiment.

Captures Customer Demand: By combining loves_count and reviews, we create a proxy for market demand that distinguishes between products people simply "wishlist" versus those they actually purchase and evaluate.

Addresses Popularity Bias: Applying a log transformation to loves_count (which ranges from 0 to over 1.4M) prevents "viral" outliers—like the Soft Pinch Liquid Blush—from skewing the model's ability to learn from standard products.

Identifies "Cult Favorites": The popularity_index (Reviews × Rating) helps the model prioritize products with high-volume, high-quality feedback, effectively filtering out 5-star items that only have a single review.

## 5) Ingredient Features (Text → Structured)

In [73]:
# 1. Calculate the number of ingredients
# We strip the brackets and quotes from the string representation before splitting
df_products['num_ingredients'] = df_products['ingredients'].apply(
    lambda x: len(str(x).replace('[', '').replace(']', '').split(',')) if x != "NA" else 0
)

# 2. Extract top 100 ingredient features using Bag-of-Words (CountVectorizer)
vectorizer = CountVectorizer(max_features=100, stop_words='english')

# Ensure we use the string version of the ingredients for the vectorizer
ingredient_text = df_products['ingredients'].astype(str)
ingredient_matrix = vectorizer.fit_transform(ingredient_text)

# 3. Convert the matrix into a DataFrame and join it back to df_products
ingredient_df = pd.DataFrame(
    ingredient_matrix.toarray(), 
    columns=[f"ingred_{name}" for name in vectorizer.get_feature_names_out()]
)

# Reset index to ensure proper alignment during concatenation
df_products = pd.concat([df_products.reset_index(drop=True), ingredient_df], axis=1)

# Verify the new columns
print(f"Total columns after ingredient engineering: {len(df_products.columns)}")
df_products[['product_name', 'num_ingredients']].head()

Total columns after ingredient engineering: 137


,product_name,num_ingredients
0,Fragrance Discovery Set,94
1,La Habana Eau de Parfum,13
2,Rainbow Bar Eau de Parfum,12
3,Kasbah Eau de Parfum,12
4,Purple Haze Eau de Parfum,12


Ingredient Analysis & Vectorization (num_ingredients, ingred_...): We transformed the raw, unstructured ingredient lists into numerical features using a combination of string parsing and natural language processing (NLP).

Ingredient Complexity as a Quality Signal: The num_ingredients count serves as a proxy for formula sophistication; in the beauty industry, a higher count or specific "hero" ingredients often correlate with premium pricing and perceived product efficacy.

Enables NLP-Based Modeling: By using CountVectorizer to extract the top 100 most frequent ingredients, we've enabled the model to identify "chemical signatures." This allows the algorithm to detect if specific components—like Hyaluronic Acid or Squalane—are statistically linked to higher customer satisfaction.

Feature Expansion: This step converted a single text column into 101 structured data points, allowing the machine learning model to "read" the label and weigh the impact of formula composition on a product's overall rating.

# Feature Transformation

## 1) Normalize numerical features (We will revise to remove the Targets, such as rating, loves_count related features, once we decided on the targets)

In [74]:
# 1. Select the numerical columns that need scaling
# We exclude binary/dummy columns (0/1) as they are already bounded
cols_to_scale = [
    'loves_count', 'rating', 'reviews', 'price_usd', 
    'price_log', 'discount_pct', 'value_ratio', 
    'log_loves', 'loves_per_price', 'popularity_index', 'num_ingredients'
]

# 2. Initialize and apply the StandardScaler
scaler = StandardScaler()
df_products[cols_to_scale] = scaler.fit_transform(df_products[cols_to_scale])

# 3. Verify the transformation (Means should be approx 0 and Std should be 1)
df_products[cols_to_scale].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
loves_count,8494.0,0.0,1.0,-0.44,-0.38,-0.29,-0.04,20.76
rating,8494.0,0.0,1.0,-6.29,-0.38,0.15,0.65,1.59
reviews,8494.0,0.0,1.0,-0.40,-0.38,-0.30,-0.03,19.18
price_usd,8494.0,0.0,1.0,-0.91,-0.50,-0.31,0.12,34.44
price_log,8494.0,0.0,1.0,-3.25,-0.60,-0.14,0.56,5.48
discount_pct,8494.0,0.0,1.0,-0.17,-0.17,-0.17,-0.17,9.93
value_ratio,8494.0,-0.0,1.0,-5.07,-0.13,-0.13,-0.13,53.21
log_loves,8494.0,-0.0,1.0,-5.08,-0.49,0.05,0.61,2.82
loves_per_price,8494.0,0.0,1.0,-0.32,-0.30,-0.24,-0.07,36.95
popularity_index,8494.0,-0.0,1.0,-0.40,-0.38,-0.30,-0.03,18.47


Feature Standardization (StandardScaler): we applied $z$-score normalization to all continuous numerical variables (e.g., price_usd, loves_count, num_ingredients) to ensure they share a common scale with a mean of 0 and a standard deviation of 1.

Ensures Model Fairness: machine learning algorithms (like Linear Regression or KNN) often misinterpret large raw numbers—such as a loves_count of 1,000,000—as being mathematically more "important" than a rating of 4.5. Scaling prevents this bias by putting all features on a level playing field.

Improves Convergence: for gradient-based models (like Neural Networks or XGBoost), having features on the same scale allows the model to find the optimal solution much faster and more reliably.

Handles Distribution Variance: by standardizing engineered features like popularity_index and loves_per_price, we make the relative "success" of a product comparable across the entire dataset, regardless of the original unit of measurement.

## 2) Category features transformation

In [75]:
# 1. Define all categorical features that provide structural meaning
categorical_cols = [
    'primary_category', 
    'secondary_category', 
    'brand_group', 
    'price_bucket'
]

# 2. Apply One-Hot Encoding across all selected features
# We include brand_group and price_bucket to capture brand equity and market tier signals
df_products = pd.get_dummies(
    df_products, 
    columns=categorical_cols, 
    drop_first=True,
    dtype=int
)

Categorical Encoding (primary_category & secondary_category): We converted the text-based category labels into a series of binary (0 or 1) columns using One-Hot Encoding.

Category-Specific Success Patterns: Product categories strongly influence performance metrics; for example, Skincare products often have higher price points and repeat purchase rates compared to Makeup, which is more trend-driven.

Eliminating the "Dummy Variable Trap": By using drop_first=True, we ensured mathematical independence between our new columns, preventing multicollinearity which can skew model coefficients.

Machine Learning Readiness: Since algorithms cannot "read" text, this transformation translates the product's department into a numerical format that allows the model to calculate the specific "weight" or impact of being in a high-growth category like Fragrance versus Bath & Body.

# Final Product Dataset

In [80]:
# We drop raw text and ID columns to keep the dataframe strictly numeric/encoded
cols_to_drop = [
    'product_id', 'product_name', 'brand_id', 'brand_name', 'size', 
    'variation_type', 'variation_value', 'variation_desc', 'ingredients', 
    'highlights', 'tertiary_category'
]

df_product_final = df_products.drop(columns=cols_to_drop)

# Verify the final shape (should be significantly wider now)
print(f"Final Product Features: {df_product_final.shape[1]}")
df_product_final.head().T

Final Product Features: 195


,0,1,2,3,4
loves_count,-0.345895,-0.383617,-0.392302,-0.395858,-0.400806
rating,-1.098350,-0.080122,0.109197,0.554352,-1.896559
reviews,-0.389139,-0.387299,-0.384538,-0.379937,-0.387299
price_usd,-0.310356,2.671043,2.671043,2.671043,2.671043
value_price_usd,35.000000,195.000000,195.000000,195.000000,195.000000
...,...,...,...,...,...
brand_group_tarte,0.000000,0.000000,0.000000,0.000000,0.000000
price_bucket_low_mid,1.000000,0.000000,0.000000,0.000000,0.000000
price_bucket_mid,0.000000,0.000000,0.000000,0.000000,0.000000
price_bucket_premium,0.000000,1.000000,1.000000,1.000000,1.000000
